In [ ]:
import numpy as np

import nd2
from skimage.io import imread, imsave
import pyclesperanto as cle
from skimage.measure import block_reduce

from pathlib import Path

import warnings

In [ ]:
# Input variables

gpu = True # Use GPU
sigma = 50 # Sigma of the Gaussian blur
raw_images = r'H:\PROJECTS-03\Pablo\Testing\test_zyla_segmentation\segmentation_test\ppp003' # Folder containing the splitted raw images as str
output_folder = r'H:\PROJECTS-03\Pablo\Testing\test_zyla_segmentation\analysis\bf_images\ppp003' # As str
img_format = 'nd2'  # Format of input images
downsample=True

In [ ]:
def path_formatting(raw_images: str, output_folder: str, img_format: str ='nd2') -> list:
    input_path = Path(raw_images)
    input_path_list = list(input_path.glob('*'+img_format))
    try:
        for input_path_file in input_path_list:
            input_path_file.resolve(strict=True)
        print('nd2 images found in {first_element}'.format(first_element=(input_path_list[0]).parent))
    except:
        print('Invalid input')

    try:
        Path(output_folder).resolve(strict=True)
    except FileNotFoundError:
        print("Output folder not found, create {output_folder}".format(output_folder=output_folder))
        Path(output_folder).mkdir()
        
    output_list_path = [Path(output_folder).resolve(strict=True) / (img_path.stem  + '.tif') for img_path in input_path_list]
    return list(map(lambda obj: obj.resolve(),input_path_list)),  list(map(lambda obj: obj.resolve(), output_list_path))

In [ ]:
def normalize_background(img: np.ndarray, sigma: int, gpu: bool = True, downsample: bool = True) -> np.ndarray:
    
    """
    This function loads an image and performs the division of the input by a blurred filtered version of itself. Give back a 8-bit clipped version of the brightfield image.
    """
    intensity_normalized = None
    copy = np.copy(img)
    copy = copy.astype(np.float32)
    
    
    
    if gpu:
        intensity_normalized = np.zeros_like(copy)
        pushed = cle.push(copy)
        cle.push(intensity_normalized)
        intensity_normalized = cle.divide_by_gaussian_background(pushed, intensity_normalized, sigma, sigma, 0)
        intensity_normalized = np.asarray(intensity_normalized) #img is pulled from GPU memory
    
    else:
        intensity_normalized = cle.divide_by_gaussian_background(copy, intensity_normalized, sigma,sigma,0)
        intensity_normalized = np.asarray(intensity_normalized)
        print(intensity_normalized.dtype)
        
    if downsample == True:
        downsampled = block_reduce(intensity_normalized, block_size=(1,2,2), func=np.mean)
        max_value = np.max(downsampled)
        min_value = np.min(downsampled)
        downsampled = (downsampled - min_value)/(max_value-min_value)*255
        downsampled = downsampled.astype(np.uint8)
        return downsampled

    else:
        print(intensity_normalized.dtype)
        max_value = np.max(intensity_normalized)
        min_value = np.min(intensity_normalized)
        intensity_normalized = (intensity_normalized - min_value)/(max_value-min_value)*255
        intensity_normalized = intensity_normalized.astype(np.uint8)
        return intensity_normalized

In [ ]:
input_path, output_list_path = path_formatting(raw_images=raw_images, output_folder=output_folder)

In [ ]:
output_list_path

In [ ]:
warnings.filterwarnings(action='ignore')
for inp, output in zip(input_path, output_list_path):
    img = nd2.imread(inp.as_posix())[:,0,:,:]
    out_img = normalize_background(img, sigma=sigma, gpu=gpu, downsample=downsample)
    imsave(output.as_posix(),out_img)

warnings.filterwarnings(action="default")